# Superluminous Supernova (SLSN-I) End-to-End Simulation

Simulates a population of magnetar-powered Type I superluminous supernovae:

| Population | Class | SED | Rate | Redshift limit | Duration |
|---|---|---|---|---|---|
| SLSN-I | `MagnetarSLSNe` | `ArnettMagnetarSpindownSED` (magnetar spin-down injected into Arnett-style diffusing ejecta, blackbody photosphere with a temperature floor; Nicholl et al. 2017) | 1/3500 of the core-collapse rate (Frohmaier et al. 2021), ~18 Gpc$^{-3}$ yr$^{-1}$ locally | z=2 | 600 d |

The SED's default priors are the sample-wide posteriors of the Nicholl et al. (2017) MOSFiT fits to 38 SLSNe-I.

This notebook is fully self-contained: it downloads the default UVEX schedule, samples a Monte Carlo
population against it, screens it down to what UVEX would actually detect, and plots the result -- using the
same pipeline the `uvex-transients` CLI runs from a config file (see `configs/full_run.yaml`, where this
population is registered as `slsne`), just driven from Python so you can poke at every intermediate object.

Run all cells top to bottom.

## Setup

Load the default schedule and configure the transient population.

In [ ]:
import numpy as np
from astropy import units as u
from matplotlib import pyplot as plt

# `uvex_transients` installs warning filters for some noisy-but-harmless dependency
# warnings (e.g. lal's Jupyter/IPython SWIG redirect notice) as soon as it's imported
# -- import it before `m4opt`/`ligo.skymap` (which trigger that warning) so the filter
# is already in place.
from uvex_transients.simulation.core import SurveySimulator
from uvex_transients.surveys import get_schedule
from uvex_transients.transients.supernovae import MagnetarSLSNe
from uvex_transients.utils.lightcurve_archive import LightcurveArchive

from m4opt.missions import uvex

schedule = get_schedule()
transients = {"slsne": MagnetarSLSNe()}
simulator = SurveySimulator(schedule, transients=transients, simulation_seed=42)

LABELS = {"slsne": "SLSN-I"}
COLORS = {"slsne": "#8172B3"}

transient = transients["slsne"]
print(
    f"redshift_limit={transient.redshift_limit}, duration_limit={transient.duration_limit}, "
    f"sed={type(transient.sed).__name__}"
)
print(f"local rate: {(transient.event_rate(0.0) * 1e9):.1f} Gpc^-3 yr^-1")

## The population model

Before running the survey, draw parameter sets from the SED's priors and look at the resulting bolometric light
curves and photospheric temperatures. Overlaid are the archival SLSN-I light curves/temperatures of Gomez et
al. (2024) (`LightcurveArchive`, `supernovae/SLSN-I`) rather than a single Nicholl et al. (2017) summary
statistic, so you can see how the full simulated population compares against the actual literature sample --
it should peak at a few $\times10^{44}$ erg/s after roughly a month and fade over hundreds of days, which is
why the duration window is long.

In [ ]:
sed = transient.sed
rng = np.random.default_rng(0)
draws = sed.sample_parameters(200, rng=rng)
t = np.geomspace(2, 1500, 200) * u.day

fig, (ax_L, ax_T) = plt.subplots(1, 2, figsize=(11, 4))
for i in range(200):
    params_i = {k: v[i] for k, v in draws.items()}
    L = sed.eval_bolometric(t, **params_i).to_value(u.erg / u.s)
    T = sed.temperature(t, **params_i).to_value(u.K)
    ax_L.loglog(t.value, L, color=COLORS["slsne"], alpha=0.1, lw=0.8)
    ax_T.loglog(t.value, T, color=COLORS["slsne"], alpha=0.1, lw=0.8)

# Overlay the archival SLSN-I bolometric light curves/temperatures (Gomez+2024) instead of a single
# Nicholl+17 summary statistic.
archive = LightcurveArchive()
events = archive.events("supernovae/SLSN-I")
for i, name in enumerate(events):
    lbol = archive.table("supernovae/SLSN-I", name, "L_bol")
    ax_L.plot(
        lbol["time"].to_value(u.day),
        lbol["L_bol"].to_value(u.erg / u.s),
        color="k",
        lw=0.5,
        alpha=0.2,
        label=f"Gomez+2024 (n={len(events)})" if i == 0 else None,
    )
    if "T_phot" in archive.fields("supernovae/SLSN-I", name):
        t_phot = archive.table("supernovae/SLSN-I", name, "T_phot")
        ax_T.plot(
            t_phot["time"].to_value(u.day),
            t_phot["T_phot"].to_value(u.K),
            color="k",
            lw=0.5,
            alpha=0.2,
        )

ax_L.axvline(transient.duration_limit.to_value(u.day), color="k", ls="--", lw=1, label="duration limit")
ax_L.set_ylim(1e41, 1e46)
ax_L.set_xlabel("Days since explosion (rest frame)")
ax_L.set_ylabel("$L_{bol}$ [erg/s]")
ax_L.legend(fontsize=8)

ax_T.axvline(transient.duration_limit.to_value(u.day), color="k", ls="--", lw=1)
ax_T.set_xlabel("Days since explosion (rest frame)")
ax_T.set_ylabel("Photospheric temperature [K]")
fig.tight_layout()

## Sample a Monte Carlo population

`generate_events` uses **windowed sampling**: it only draws events within HEALPix pixels/time
bins the schedule could plausibly have caught, given the transient's own duration window. SLSNe are rare, so no
downsampling is applied.

In [ ]:
TIME_BINS = 200
NSIDE = 256
DOWNSAMPLE = None
SCALE = DOWNSAMPLE or 1  # every count below is multiplied back up by the downsampling factor

catalog = simulator.generate_events(time_bins=TIME_BINS, nside=NSIDE, downsample=DOWNSAMPLE)
print(f"Sampled {len(catalog) * SCALE:,} events across {TIME_BINS} time bin(s) at NSIDE={NSIDE}.")

## Screen the population

Two progressively more expensive passes narrow the freshly-sampled catalog down to what matters:
first, a cheap check of whether an event could *ever* clear a fixed magnitude limit
(`filter_by_limiting_magnitude`); then the real question of whether it was actually detected
above a given SNR by an observation the schedule made (`filter_by_snr`).

In [ ]:
MAG_LIMIT = 25.0
SNR_THRESHOLD = 5.0

mag_filtered = simulator.filter_by_limiting_magnitude(catalog, uvex, mag_limit=MAG_LIMIT)
print(f"{len(mag_filtered) * SCALE:,} could ever clear {MAG_LIMIT} AB mag.")

detected = simulator.filter_by_snr(mag_filtered, uvex, snr_threshold=SNR_THRESHOLD)
print(f"{len(detected) * SCALE:,} were detected above SNR={SNR_THRESHOLD}.")

## Detection funnel

In [ ]:
stage_names = ["Sampled", f"Mag < {MAG_LIMIT}", f"SNR > {SNR_THRESHOLD}"]
counts = [len(stage) * SCALE for stage in (catalog, mag_filtered, detected)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(stage_names, counts, color=COLORS["slsne"])
for i, count in enumerate(counts):
    ax.text(i, count, f"{count:,}", ha="center", va="bottom", fontsize=8)
ax.set_yscale("log")
ax.set_ylabel("Number of events")
ax.set_title("Detection funnel")
fig.tight_layout()

print(f"Detected fraction of sampled events: {counts[-1] / max(counts[0], 1):.2%}")

## Yield summary

Tabulate the population's footprint-aware exposure (`SurveySimulator.compute_effective_exposure`,
an `ExposureCatalog`), then combine it with `catalog`/`detected` into a yield estimate
(`EventCatalog.compute_yield_summary`, a `YieldTable`) -- rate, intrinsic UVEX event count,
detection probability, and expected detections, each with both Clopper-Pearson (Monte Carlo) and
rate-normalization confidence bounds. This is the same summary the `uvex-transients run` CLI
command writes to `exposure.ecsv`/`yield_summary.ecsv`/`yield_summary.txt`.
`YieldTable.display_expected_detections` renders just the expected-detections estimate, with both
uncertainties stacked, as typeset LaTeX.

In [ ]:
exposure = simulator.compute_effective_exposure(time_bins=TIME_BINS, nside=NSIDE)
yield_table = catalog.compute_yield_summary(detected, exposure, transients)
yield_table.display_expected_detections()

## Sky distribution

In [ ]:
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111, projection="aitoff")
ax.grid(True)

ax.scatter(
    catalog.coord.ra.wrap_at(180 * u.deg).radian,
    catalog.coord.dec.radian,
    s=4,
    alpha=0.3,
    color="#888888",
    label=f"Sampled ({SCALE * len(catalog):,})",
)
ax.scatter(
    detected.coord.ra.wrap_at(180 * u.deg).radian,
    detected.coord.dec.radian,
    s=12,
    color=COLORS["slsne"],
    label=f"Detected ({SCALE * len(detected):,})",
)
ax.legend(loc="lower right", markerscale=2, fontsize=7)
ax.set_title("Sky distribution")
fig.tight_layout()

## Redshift distribution

SLSNe are luminous and UV-bright, so unlike most populations a large share of the detected events sit at
high redshift, out towards the sampling limit.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
bins = np.linspace(0, transient.redshift_limit, 30)
ax.hist(catalog.redshift, bins=bins, color="#888888", label=f"Sampled ({len(catalog)})")
ax.hist(detected.redshift, bins=bins, color=COLORS["slsne"], label=f"Detected ({len(detected)})")
ax.set_yscale("log")
ax.set_xlabel("Redshift")
ax.set_ylabel("Number of events (before rescaling)")
ax.legend()
fig.tight_layout()

## Example light curves

Reconstruct one event as a real `Event` (`EventCatalog.get_events`) and run full synthetic photometry
(`Event.simulate_photometry`) against every observation the schedule actually made of it. Prefer a detected
event, falling back to the best available stage if none was detected. Filled squares are detections; open
triangles are upper limits; the lines are the noiseless model.

In [ ]:
def plot_example(key, seed=1):
    """Plot one reconstructed event of population `key`, drawn from the deepest stage that has one."""
    rng = np.random.default_rng(seed)
    for label, source in (("detected", detected), ("mag-screened", mag_filtered), ("sampled", catalog)):
        ids = source.event_id[source.transient_type == key]
        if len(ids) > 0:
            stage = label
            break
    else:
        raise RuntimeError(
            f"No {key} events at all were sampled -- try a larger TIME_BINS/NSIDE or a smaller DOWNSAMPLE."
        )

    event = source.get_events(int(rng.choice(ids)), transients, schedule)
    print(f"Example {LABELS[key]} event (from the {stage!r} stage):")
    print(event)

    phot = event.simulate_photometry(uvex)
    t_since_explosion = (phot["obs_time"] - event.t_explosion).to(u.day)
    t_theory = np.linspace(0, transients[key].duration_limit.to_value(u.day), 300) * u.day

    fig, ax = plt.subplots(figsize=(7, 4))
    for band, color in {"FUV": "#4C72B0", "NUV": "#DD8452"}.items():
        ax.plot(t_theory.value, event.mag(t_theory, uvex, band=band).value, color=color, lw=1.5, alpha=0.6)

        in_band = np.isfinite(phot["ab_mag"]) & (phot["band"] == band)
        detected_pts = in_band & (phot["snr"] > SNR_THRESHOLD)
        upper_limits = in_band & (phot["snr"] <= SNR_THRESHOLD)

        if np.any(detected_pts):
            ax.errorbar(
                t_since_explosion[detected_pts].value,
                phot["ab_mag"][detected_pts],
                yerr=5 * phot["mag_err"][detected_pts],
                marker="s",
                mfc=color,
                mec="k",
                ecolor=color,
                linestyle="none",
                label=band,
            )
        if np.any(upper_limits):
            ax.errorbar(
                t_since_explosion[upper_limits].value,
                phot["ab_mag"][upper_limits],
                yerr=[
                    phot["mag_upper"][upper_limits] - phot["ab_mag"][upper_limits],
                    np.abs(phot["mag_lower"][upper_limits] - phot["ab_mag"][upper_limits]),
                ],
                marker="v",
                mfc="w",
                mec=color,
                ecolor=color,
                linestyle="none",
            )

    ax.invert_yaxis()
    ax.set_xlabel("Days since explosion")
    ax.set_ylabel("AB magnitude")
    ax.set_title(f"{LABELS[key]}: event {event.event_id} (z={event.redshift:.3f}, {event.n_observations} observations)")
    ax.set_ylim([30, None])
    ax.legend()
    fig.tight_layout()
    plt.show()

### SLSN-I

In [ ]:
plot_example("slsne")